# TN2 — Tầm nhìn DS-TCN 192 kênh, 4 fold

## Vì sao chạy thang này

### Kết quả ĐÃ CÓ của bản 64 kênh — notebook này KHÔNG chạy lại

Vòng sàng lọc một fold trên 64 kênh cho kết quả đơn điệu — ba cấu hình
notebook này chạy:

| kernel | tầm nhìn | điểm `val_KL` | train_mse | train_pearson |
|---:|---:|---:|---:|---:|
| 5 | 121 | **0,8458** | 0,02127 | 0,5851 |
| 7 | 181 | 0,8176 | 0,02059 | 0,5921 |
| 9 | 241 | 0,8041 | 0,01939 | 0,6041 |

Tương quan tầm nhìn với điểm: **−0,983**. Tầm nhìn càng dài, chọn kênh càng tệ.

Hai cấu hình tầm nhìn dài hơn cũng đã chạy và **tệ hơn cả ba dòng trên**:
tầm nhìn 301 được 0,7940 và tầm nhìn 361 được 0,7759. Bỏ khỏi vòng này.

Và cùng lúc điểm tụt thì model **dự báo giỏi lên** — `train_mse` giảm 13%,
`train_pearson` tăng. Model càng giỏi dự báo càng chọn kênh dở.

Giả thuyết: tiêu chí chọn kênh là "ứng viên nào tự dự báo được chính nó tốt
nhất". Model tầm nhìn ngắn chỉ đoán giỏi sóng **thật sự tuần hoàn**; model tầm
nhìn dài đoán giỏi **mọi sóng trơn**, kể cả kênh nhiễu có cấu trúc. Tầm nhìn
ngắn hoạt động như bộ lọc — dở đúng chỗ cần dở.

**Notebook này xem xu hướng đó có lặp lại ở 192 kênh không.** Nếu có, giả thuyết
mạnh hơn hẳn: hai bề rộng kênh khác nhau 8 lần mà cùng một xu hướng.

## Chạy gì

Ba cấu hình, **đủ bốn fold**, một seed. Bỏ hai tầm nhìn dài nhất vì bản 64 kênh
đã cho thấy chúng tệ nhất.

| kernel | tầm nhìn | tham số | phủ cửa sổ 200 |
|---:|---:|---:|---:|
| 5 | 121 | 310.873 | 60% |
| 7 | 181 | 313.945 | 90% |
| 9 | 241 | 317.017 | 100% |

Điểm nối đầu thang, đã có: **kernel 3, tầm nhìn 61, 307.801 tham số**.

## Thời gian

Tuỳ đã chạy vòng sàng lọc `TN2_ReceptiveField_DS_TCN_c192` chưa:

    đã chạy      fold val_KL được bỏ qua  ->  9 fold  ->  khoảng 1,8 giờ
    chưa chạy    làm đủ 12 fold           ->            khoảng 2,4 giờ

Ô khôi phục ở mục 1 tự lo phần này — nó kéo mọi kết quả `tn2_rf` đã có từ Drive
về, và `run_cv.py` bỏ qua fold nào đã xong.

## Chưa kết luận được sau vòng này

Một seed. `seed_std` của tám cấu hình TN1 trải từ 0,0007 tới 0,0108, mỗi kiến
trúc một khác — không mượn của nhau được.

Vòng này trả lời câu hẹp hơn: **xu hướng ở 64 kênh có lặp lại ở 192 kênh
không**, và **có giữ nguyên trên cả bốn fold không**. Nếu `val_DF` — fold khó
nhất — cho thứ tự ngược thì kết luận là do fold chứ không do tầm nhìn.

## Mốc để đặt cạnh

| | tham số | tầm nhìn | cv_score |
|---|---:|---:|---:|
| DS-TCN-192 k3n4 no_norm do0.2 | 307.801 | **61** | 0,762714 *(1 seed)* |
| DS-TCN-64 k3n4 no_norm do0.2 | 37.081 | **61** | 0,760878 ± 0,003095 *(3 seed)* |
| LSTM-352 | 1.502.713 | — | 0,756992 ± 0,004156 |

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn.

In [2]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 85ccc08
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


Lấy `by_user/` và `windows/` từ Drive.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Khôi phục kết quả `tn2_rf` đã có.

**Bước này quyết định notebook chạy 1,8 giờ hay 2,4 giờ.** Chưa chạy vòng sàng
lọc thì nó in `fold đã có: 0` và làm đủ mười hai fold — vẫn đúng, chỉ lâu hơn.

In [ ]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn2_rf_*c192*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm ba bản cài đặt

Số tham số phải ra đúng 310.873 / 313.945 / 317.017.

**Đọc dòng cuối mỗi lệnh.** Phải là `TẤT CẢ ĐẠT`.

In [5]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   310873

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 5, 4 khối -> tầm nhìn 121, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 121/200 mẫu gần nhất, mất 40% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT

In [6]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   313945

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 7, 4 khối -> tầm nhìn 181, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 181/200 mẫu gần nhất, mất 9% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT 

In [7]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   317017

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 9, 4 khối -> tầm nhìn 241, cửa sổ vào 200
   phủ trọn cửa sổ                                            đạt

TẤT CẢ ĐẠT — bản cài đặt dùng được.


## 3. Chạy đủ 4 fold, một seed

Không có `--folds` nên chạy đủ bốn. Fold nào đã có sẽ in
`đã có kết quả 0.xxxx — bỏ qua, không train lại`.

**kernel 5 — tầm nhìn 121, 310.873 tham số**

In [8]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03603  pearson 0.4361   0.4 phút
epoch  1  mse 0.02365  pearson 0.5273   0.8 phút
epoch  2  mse 0.02250  pearson 0.5460   1.1 phút
epoch  3  mse 0.02197  pearson 0.5552   1.5 phút
epoch  4  mse 0.02160  pearson 0.5608   1.9 phút
epoch  5  mse 0.02131  pearson 0.5651   2.2 phút
epoch  6  mse 0.02112  pearson 0.5691   2.6 phút
epoch  7  mse 0.02092  pearson 0.5715   3.0 phút
epoch  8  mse 0.02076  pearson 0.5749   3.3 phút
epoch  9  mse 0.02060  pearson 0.5775   3.7 phút
epoch 10  mse 0.02043  pearson 0.5800   4.1 phút
epoch 11  mse 0.02029  pearson 0.5815   4.5 phút
epoch 12  mse 0.02022  pearson 0.5829   4.8 phút
epoch 13  mse 0.02013  pearson 0.5845   5.2 phút
epoch 14  mse 0.01996  pearson 0.5876   5.6 phút
epoch 15  mse 0.01987  pearson 0.5882 

**kernel 7 — tầm nhìn 181, 313.945 tham số**

In [9]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 7 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03535  pearson 0.4479   0.4 phút
epoch  1  mse 0.02314  pearson 0.5327   0.8 phút
epoch  2  mse 0.02194  pearson 0.5500   1.2 phút
epoch  3  mse 0.02139  pearson 0.5588   1.6 phút
epoch  4  mse 0.02099  pearson 0.5653   2.0 phút
epoch  5  mse 0.02070  pearson 0.5703   2.4 phút
epoch  6  mse 0.02042  pearson 0.5747   2.8 phút
epoch  7  mse 0.02012  pearson 0.5789   3.2 phút
epoch  8  mse 0.01991  pearson 0.5825   3.6 phút
epoch  9  mse 0.01969  pearson 0.5845   4.0 phút
epoch 10  mse 0.01954  pearson 0.5863   4.4 phút
epoch 11  mse 0.01938  pearson 0.5891   4.8 phút
epoch 12  mse 0.01920  pearson 0.5920   5.2 phút
epoch 13  mse 0.01904  pearson 0.5938   5.6 phút
epoch 14  mse 0.01888  pearson 0.5960   6.0 phút
epoch 15  mse 0.01870  pearson 0.5985 

**kernel 9 — tầm nhìn 241, 317.017 tham số**

In [10]:
!python scripts/run_cv.py --experiment tn2_rf --model ds_tcn --channels 192 \
    --kernel_size 9 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn2_rf  -> runs/tn2_rf/
cấu hình ds_tcn_c192_k9_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  NVIDIA L4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.03565  pearson 0.4432   0.4 phút
epoch  1  mse 0.02287  pearson 0.5359   0.9 phút
epoch  2  mse 0.02155  pearson 0.5559   1.3 phút
epoch  3  mse 0.02084  pearson 0.5671   1.7 phút
epoch  4  mse 0.02031  pearson 0.5744   2.1 phút
epoch  5  mse 0.01980  pearson 0.5818   2.6 phút
epoch  6  mse 0.01918  pearson 0.5894   3.0 phút
epoch  7  mse 0.01874  pearson 0.5962   3.4 phút
epoch  8  mse 0.01839  pearson 0.6006   3.8 phút
epoch  9  mse 0.01805  pearson 0.6054   4.2 phút
epoch 10  mse 0.01776  pearson 0.6086   4.7 phút
epoch 11  mse 0.01756  pearson 0.6127   5.1 phút
epoch 12  mse 0.01734  pearson 0.6151   5.5 phút
epoch 13  mse 0.01718  pearson 0.6173   5.9 phút
epoch 14  mse 0.01701  pearson 0.6195   6.4 phút
epoch 15  mse 0.01688  pearson 0.6204 

## 4. Cất kết quả

In [11]:
!python scripts/save_results.py tn2_rf --out tn2_rf_c192_4fold

runs/tn2_rf/  ->  runs/tn2_rf_c192_4fold.zip   (14.1 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-07 21:58   tn2_rf/
        0  2026-09-07 19:08   tn2_rf/ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 19:23   tn2_rf/ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 19:38   tn2_rf/ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 19:52   tn2_rf/ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 20:05   tn2_rf/ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 20:21   tn2_rf/ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 20:37   tn2_rf/ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 20:51   tn2_rf/ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 21:05   tn2_rf/ds_tcn_c192_k9_n4_none_do0.2_dpel_mse_corr0.

## 5. Bảng so

Đủ bốn fold nên dòng `TONG` có, `compare_cv` hiện `cv_score` thật. Bảng này gồm
cả các cấu hình 64 kênh nếu chúng cũng nằm trong `runs/tn2_rf/`.

In [12]:
!python scripts/compare_cv.py --experiment tn2_rf


BẢNG 1 — cv_score, thực nghiệm tn2_rf
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_corr0.9    310873     1   0.764428       N/A  0.089609   s0 0.7644
ds_tcn_c192_k9_n4_none_do0.2_dpel_mse_corr0.9    317017     1   0.736623       N/A  0.070749   s0 0.7366
ds_tcn_c192_k7_n4_none_do0.2_dpel_mse_corr0.9    313945     1   0.732562       N/A  0.075568   s0 0.7326

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 6. Xu hướng có lặp lại không

In điểm từng fold. Hai câu cần trả lời:

1. Ở 192 kênh, thứ tự có giống 64 kênh không — tầm nhìn ngắn hơn thì điểm cao hơn?
2. Thứ tự đó có giữ nguyên ở cả bốn fold, hay chỉ đúng ở `val_KL`?

In [13]:
import csv, re
RF = {3: 61, 5: 121, 7: 181, 9: 241, 11: 301, 13: 361}
rows = [r for r in csv.DictReader(open("runs/tn2_rf/summary.csv"))
        if "_c192_" in r["run_id"] and r["fold"] != "TONG"]
for r in sorted(rows, key=lambda r: (r["fold"], r["run_id"])):
    k = int(re.search(r"_k(\d+)_", r["run_id"]).group(1))
    print(" ", r["fold"], " kernel", k, " tầm nhìn", RF[k], " ", r["score_macro"])

  val_AB  kernel 5  tầm nhìn 121   0.8012105066583294
  val_AB  kernel 7  tầm nhìn 181   0.7856542046969223
  val_AB  kernel 9  tầm nhìn 241   0.7941171127476142
  val_CE  kernel 5  tầm nhìn 121   0.8032210680348648
  val_CE  kernel 7  tầm nhìn 181   0.768966870860248
  val_CE  kernel 9  tầm nhìn 241   0.7770199573161753
  val_DF  kernel 5  tầm nhìn 121   0.6117364226623185
  val_DF  kernel 7  tầm nhìn 181   0.6021006574731258
  val_DF  kernel 9  tầm nhìn 241   0.6159380166363821
  val_KL  kernel 5  tầm nhìn 121   0.8415421449743689
  val_KL  kernel 7  tầm nhìn 181   0.7735247893759685
  val_KL  kernel 9  tầm nhìn 241   0.7594178378536487


## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()